# Check matchs repu et comparaison résultats vs ND
(par pnum vs id_syceron ET par check snippets de textes)

In [21]:
import pandas as pd
import re

PATH_LISTE_PAYS = "../data/raw/liste_pays_republique_stable.txt"

df = pd.read_csv("../data/interim/2_4_interventions_nettoyees.csv", low_memory=False)

df_extract = pd.read_csv(
    "../data/interim/1_2_extract_15_16_concat.csv", low_memory=False
)

df_ND1516 = pd.read_csv(
    "../data/raw/test_regards_citoyens/ND15+16_interventions_hemicycle_rich.tsv",
    sep="\t",
    engine="python",
    on_bad_lines="warn",
)


## Nettoyage du texte :
déjà fait en amont dans notre regroupement mais utile pour autres df


In [22]:
# Trace fonction nettoyage (voir 2_1_filtrage.py)
import unicodedata
import html


def nettoyer_texte(texte):
    if not isinstance(texte, str):
        return ""
    # Normaliser les caractères Unicode
    texte = unicodedata.normalize("NFC", texte)
    # Décoder les entités HTML
    texte = html.unescape(texte)
    # Supprimer les balises HTML/XML > espace (éviter collage de mots)
    texte = re.sub(r"<[^>]+>", " ", texte)
    # Supprimer contenu entre parenthèses
    # NOTE : CHOIX FORT SELON CE QUI VEUT ÊTRE ÉTUDIÉ
    # Supprime des didascalies ("Applaudissements", etc.)
    # mais aussi tout autre contenu entre parenthèses
    # ne gère pas les parenthèses imbriquées mais sont extrêmement rares (parfois sur (e))
    texte = re.sub(r"\([^()]*\)", "", texte)
    # Uniformiser apostrophes (utile pour regex)
    texte = texte.replace("’", "'").replace("\u02bc", "'")
    # Normaliser les espaces (après unescape(), couvre \xa0, \t, \n)
    # et supprimer les espaces multiples
    texte = re.sub(r"\s+", " ", texte).strip()

    return texte


## Regex
Logique de l'identification des mentions valides de République :
- regex sur le champ lexical "républi"
- mais exclusion de certains termes (positions, pas de chaînage) car les
termes exclus peuvent apparaître aussi avec les termes voulus (les idées
républicaines sont menacées par Les Républicains) : chaîner risquerait
de virer des occurrences qu'on aurait voulu garder.

In [23]:
# préparer les pays à exclure
with open(PATH_LISTE_PAYS, "r", encoding="utf-8") as f:
    liste_pays = [line.strip() for line in f]

# pattern regex pour les pays, rendu non capturant plus bas
pattern_pays = r"|".join(re.escape(p) for p in liste_pays)

# Regex de la famille du mot République (simplifié ici)
pattern_lexical = re.compile(
    r"républi",  # même au milieu des mots
    re.I,
)

# Regex des expressions à exclure
# logique : groupes (?:…) non capturant, utilisés juste pour les positions

# Expressions à exclure - casse exacte
# possible cas du féminin… mais pas d'occurrence dans la base avec nos exclusions
pattern_excl_case_sensitive = re.compile(
    # --- Exclusion des occurrences liées au parti les Républicains ---
    r"(?:\b[LlDd]es Républicains\b)"  # garde la casse pour identifier le parti (et pas un adjectif)
    r"|(?:\baux Républicains\b)"  # idem majuscule pour le groupe
    r"|(?:\b[Cc]ollègues? Républicains?\b)"  # cas avec et sans maj pour collègues
    r"|(?:\bsénateurs? Républicains?\b)"  # pas de maj sénateurs ou féminin dans la base après exclu, mais aviser
    r"|(?:\bdéputés? Républicains?\b)"  # pas de maj députés ou féminin dans la base après exclu, mais aviser
    # --- Spécifique corpus AN choisi ---
    r"|(?:\bLes Républicain\b)"  # typo manque s = spécifique corpus AN (4 occurrences)
    r"|(?:\bentre Républicains?\b)"  # spécifique corpus AN (4 occurrences)
    r"|(?:\bex-Républicains?\b)"  # spécifique corpus AN (4 occurrences)
    r"|(?:\banciens? Républicains?\b)"  # spécifique corpus AN (1 occurrence)
    r"|(?:\bseuls Républicains\b)"  # spécifique corpus AN (1 occurrence)
    r"|(?:\bparlementaires Républicains\b)"  # spécifique corpus AN (1 occurrence)
    r"|(?:\bélus Républicains\b)"  # spécifique corpus AN (1 occurrence)
    r"|(?:\bgroupeLes Républicains\b)"  # spécifique corpus AN (1 occurrence)
    r"|(?:\b[Nn]ous Républicains\b)"  # spécifique corpus AN (2 occurrences)
    r"|(?:\bcertains Républicains\b)"  # spécifique corpus AN (4 occurrences)
    r"|(?:\bamis Républicains\b)"  # spécifique corpus AN (1 occurrence)
    # --- Occurrences plusieurs partis ---
    r"|(?:\bdroite, Républicains et macronistes\b)"  # spécifique corpus AN (1 occurrence)
    r"|(?:\bRépublicains-Front national\b)"  # spécifique corpus AN (1 occurrence)
    r"|(?:\bMacronistes, Républicains, lepénistes\b)"  # spécifique corpus AN (1 occurrence)
    r"|(?:\bRassemblement national, Républicains et macronistes\b)"  # spécifique corpus AN (1 occurrence)
    r"|(?:\bparti Républicain\b)"  # spécifique corpus AN (1 occurrence : US)
    # --- Titres de presse ---
    r"|(?:\bL[’']Est républicain\b)"  # le journal
    r"|(?:\bLa Nouvelle République\b)"  # le journal
)

# Expressions à exclure - ignorer la casse
pattern_excl_case_insensitive = re.compile(
    # --- Partis et groupes politiques ---
    # TODO : confirmation MATTHIAS POUR EXCLUSIONS NV CAS PARTIS
    r"(?:\bgauche démocrate et républicaine\b)"  # premier sans |
    r"|(?:\bGauche démocrate républicaine\b)"  # (feinte) ajout léo
    r"|(?:\bGauche démocratique et Républicaine\b)"  # (feinte) ajout léo
    r"|(?:\bgauche démocrate et républicaine-NUPES\b)"
    r"|(?:\bsocialiste, écologiste et républicain\b)"
    r"|(?:\bgroupe socialiste et républicain\b)"  # ajout léo (garder groupe pour limiter flag)
    r"|(?:\bcommuniste républicain citoyen et écologiste\b)"  # ajout léo
    r"|(?:\brépublique en marche\b)"
    r"|(?:\bconstructifs : républicains, UDI, indépendants\b)"  # ajout léo
    r"|(?:\bconstructifs : républicains, UDI et apparentés\b)"  # (feinte) ajout léo
    r"|(?:\bLes Indépendants - République et Territoires\b)"  # ajout léo
    r"|(?:\bLes Indépendants-République et Territoires\b)"  # (feinte) ajout léo
    # TODO : aviser
    # "Rassemblement pour la République" RPR 1 cas -> mais risque appel rassemblement sensible casse ?
    # NOTE : ont également été testés (0 cas ici, mais voir selon autres législatures)
    # "Union des démocrates pour la République" UDR  / "Union des droites pour la République"
    # "Union pour une Nouvelle République" / "Debout la République"
    # "Forum des républicains sociaux" / "Identité et République"
    # --- Fonctions et institutions ---
    # TODO : MATTHIAS CHOISI POUR exclusion présidente(s) de la république
    r"|(?:\bprésidents? de la république\b)"
    r"|(?:\bprésidentes? de la république\b)"  # 7 cas pour féminiser la fonction ou souhaiter élection MLP
    r"|(?:\bprésidences? de la république\b)"
    r"|(?:\bprocureurs? de la république\b)"
    r"|(?:\bcours? de justice de la république\b)"  # nb : cours de sûreté est lui gardé car projet loi LR et pas une institution
    r"|(?:\badministration générale de la république\b)"
    r"|(?:\bgouvernement de la république française\b)"  # pas de pluriel dans corpus
    r"|(?:\bInstitut supérieur des langues de la République française\b)"
    r"|(?:\bHaut-commissariat de la République\b)"  # préfets en Kanaky et Polynésie française uniquement
    r"|(?:\bHaut-commissaire de la République\b)"  # ibid
    r"|(?:\bcompagnies? républicaines? de sécurité\b)"
    r"|(?:\bgarde républicaine\b)"
    r"|(?:\bgardes? républicains?\b)"
    r"|(?:\buniversités? de la République\b)"  # sur corpus 2017-2024 réf à une commission d'enquête
    r"|(?:\binstitut Famille et République\b)"  # institut privé de la galaxie LMPT
    # --- Titres de lois ---
    r"|(?:\bnouvelle organisation territoriale de la République\b)"
    r"|(?:\bconfortant le respect des principes de la République\b)"
    r"|(?:\bpour une république numérique\b)"
    # --- Expression et législations ---
    # NOTE: Le choix a été fait de ne pas exclure ces formes qui sont pertinentes à garder
    # Elles sont conservées ici en commentaires pour traçabilité
    # r"|(?:\bcontrat d[’']engagement républicain\b)"
    # r"|(?:\bcontrat d[’']engagement au respect des principes de la République\b)"
    # r"|(?:\bcontrat d[’']intégration républicaine\b)"
    # r"|(?:\bquartier[s]? de reconquête républicaine\b)"
    # --- Lieux (places, monuments) ---
    r"|(?:\bplace de la République\b)"  # (45 occurrences)
    # --- Pays, territoires, entités, etc. ---
    r"|(?:\brépubliques? soviétiques?\b)"
    r"|(?:\bex-républiques? soviétiques?\b)"
    r"|(?:\brépublique de Weimar\b)"
    r"|(?:\bRépublique yougoslave\b)"  # spécifique corpus AN
    r"|(?:\brépublique du Haut-Karabakh\b)"  # spécifique corpus AN
    r"|(?:\brépublique d[’']Artsakh\b)"  # spécifique corpus AN
    r"|(?:\brépublique de l[’']Artsakh\b)"  # spécifique corpus AN
    r"|(?:\brépublique des Fidji\b)"  # spécifique corpus AN
    r"|(?:\bRépublique de Chine\b)"  # spécifique corpus AN
    r"|(?:\brépubliques? du Donbass\b)"  # spécifique corpus AN
    r"|(?:\brépublique de Crimée\b)"  # spécifique corpus AN
    r"|(?:\bRépublique démocratique d[’']Arménie\b)"  # spécifique corpus AN
    r"|(?:\bRépubliques du Bénin et du Sénégal\b)"  # spécifique corpus AN
    r"|(?:\b(?:" + pattern_pays + r")\b)",  # ajout des exclusions de pays (liste)
    re.I,
)

# TODO / NOTE : quelques (~10) "république islamique" sans précision pour parler de l'Iran
# mais risque de supprimer d'autres occurrences que l'on veut garder,
# ou alors aviser majuscule a République vs sans ? -> trop niche


# Fonction de décompte des occurrences
def count_lexical_outside_excl(text):
    """
    Compte les occurrences valides du champ lexical "républi" (hors zones
    d'exclusion). Early-exit via pattern_lexical.search() avant de calculer
    les positions d'exclusion (coûteux, notamment la liste de pays) : utile
    car la grande majorité des textes ne contiennent aucune occurrence.

    NOTE : l'ancienne fonction contains_lexical_outside_excl() a été
    supprimée (07/07/2026) : elle est strictement équivalente à
    (count_lexical_outside_excl(text) > 0), donc redondante. Équivalence
    vérifiée par test, même nombre de matchs.

    NOTE : on pourrait optimiser in_excl() via spans triés + bisect, pas
    indispensable ici et plus complexe.
    + probablement pas rentable car pas assez occurrences par texte ?
    """
    if pd.isna(text) or not pattern_lexical.search(text):
        return 0
    # Trouver les positions des expressions exclues
    excl_positions = [m.span() for m in pattern_excl_case_sensitive.finditer(text)] + [
        m.span() for m in pattern_excl_case_insensitive.finditer(text)
    ]

    # Fonction pour vérifier si une position est dans une zone exclue
    def in_excl(pos):
        for start, end in excl_positions:
            if start <= pos < end:
                return True
        return False

    return sum(1 for m in pattern_lexical.finditer(text) if not in_excl(m.start()))


## Application

In [24]:
### df regroupé

print("-----------------")
print("--- df (fusion) ---")
print("-----------------")

# ========== Appliquer comptage et identification mentions valides ==========
df["nombre_mentions_repu"] = df["texte"].apply(count_lexical_outside_excl)
df["repu_match_valide"] = df["nombre_mentions_repu"] > 0
print(df["repu_match_valide"].value_counts())
print(df.shape)

# check avec le texte brut du df regroupé au cas ou
print("-----------------")

# ========== Appliquer comptage et identification mentions valides ==========
df["nombre_mentions_repu_texte_brut"] = df["texte_brut"].apply(
    count_lexical_outside_excl
)
df["repu_match_valide_texte_brut"] = df["nombre_mentions_repu_texte_brut"] > 0
print(df["repu_match_valide_texte_brut"].value_counts())
print(df.shape)

-----------------
--- df (fusion) ---
-----------------
repu_match_valide
False    506192
True      10831
Name: count, dtype: int64
(517023, 68)
-----------------
repu_match_valide_texte_brut
False    505725
True      11298
Name: count, dtype: int64
(517023, 70)


In [25]:
### df_extract sans regroupement
print("-----------------")
print("--- df_extract ---")
print("-----------------")

# ========== Appliquer comptage et identification mentions valides ==========
df_extract["nombre_mentions_repu"] = df_extract["texte"].apply(
    count_lexical_outside_excl
)
df_extract["repu_match_valide"] = df_extract["nombre_mentions_repu"] > 0
print(df_extract["repu_match_valide"].value_counts())
print(df_extract.shape)


### df_extract sans regroupement mais nettoyage
print("-----------------")

# ========== Appliquer comptage et identification mentions valides ==========
df_extract["nombre_mentions_repu_net"] = (
    df_extract["texte"].apply(nettoyer_texte).apply(count_lexical_outside_excl)
)
df_extract["repu_match_valide_net"] = df_extract["nombre_mentions_repu_net"] > 0
print(df_extract["repu_match_valide_net"].value_counts())
print(df_extract.shape)

-----------------
--- df_extract ---
-----------------
repu_match_valide
False    1114011
True       13818
Name: count, dtype: int64
(1127829, 39)
-----------------
repu_match_valide_net
False    1114734
True       13095
Name: count, dtype: int64
(1127829, 41)


In [26]:
### df_ND1516

print("-----------------")
print("--- df_ND1516 ---")
print("-----------------")


# ========== Appliquer comptage et identification mentions valides ==========
df_ND1516["nombre_mentions_repu"] = df_ND1516["intervention"].apply(
    count_lexical_outside_excl
)
df_ND1516["repu_match_valide"] = df_ND1516["nombre_mentions_repu"] > 0
print(df_ND1516["repu_match_valide"].value_counts())
print(df_ND1516.shape)


### df_ND1516 mais avec nettoyage
print("-----------------")

# ========== Appliquer comptage et identification mentions valides ==========
df_ND1516["nombre_mentions_repu_net"] = (
    df_ND1516["intervention"].apply(nettoyer_texte).apply(count_lexical_outside_excl)
)
df_ND1516["repu_match_valide_net"] = df_ND1516["nombre_mentions_repu_net"] > 0
print(df_ND1516["repu_match_valide_net"].value_counts())
print(df_ND1516.shape)


-----------------
--- df_ND1516 ---
-----------------
repu_match_valide
False    1377532
True       13675
Name: count, dtype: int64
(1391207, 18)
-----------------
repu_match_valide_net
False    1377532
True       13675
Name: count, dtype: int64
(1391207, 20)


#### Test sans les cas intervenants manquants

In [27]:
# Cas extraction brute
mask_extract_speaker = (
    df_extract["id_acteur"].notna()
    | df_extract["id_orateur"].notna()
    | df_extract["nom_orateur"].fillna("").astype(str).str.strip().ne("")
)
df_extract_with_speaker = df_extract[mask_extract_speaker].copy()

print("-----------------")
print("--- df_extract_with_speaker ---")
print("-----------------")
print("df_extract_with_speaker shape : ", df_extract_with_speaker.shape)
print("-----------------")
print(
    "df_extract_with_speaker repu_match_valide counts :\n",
    df_extract_with_speaker["repu_match_valide"].value_counts(),
)
print("-----------------")
print(
    "df_extract_with_speaker repu_match_valide_net counts :\n",
    df_extract_with_speaker["repu_match_valide_net"].value_counts(),
)

# cas nd
mask_ND_speaker = df_ND1516["parlementaire"].fillna("").astype(str).str.strip().ne(
    ""
) | df_ND1516["personnalite"].fillna("").astype(str).str.strip().ne("")
df_ND1516_with_speaker = df_ND1516[mask_ND_speaker].copy()
print("-----------------")
print("--- df_ND1516_with_speaker ---")
print("-----------------")
print("df_ND1516_with_speaker shape : ", df_ND1516_with_speaker.shape)
print("-----------------")
print(
    "df_ND1516_with_speaker repu_match_valide counts :\n",
    df_ND1516_with_speaker["repu_match_valide"].value_counts(),
)
print("-----------------")
print(
    "df_ND1516_with_speaker repu_match_valide_net counts :\n",
    df_ND1516_with_speaker["repu_match_valide_net"].value_counts(),
)


-----------------
--- df_extract_with_speaker ---
-----------------
df_extract_with_speaker shape :  (1027685, 41)
-----------------
df_extract_with_speaker repu_match_valide counts :
 repu_match_valide
False    1013868
True       13817
Name: count, dtype: int64
-----------------
df_extract_with_speaker repu_match_valide_net counts :
 repu_match_valide_net
False    1014590
True       13095
Name: count, dtype: int64
-----------------
--- df_ND1516_with_speaker ---
-----------------
df_ND1516_with_speaker shape :  (1088105, 20)
-----------------
df_ND1516_with_speaker repu_match_valide counts :
 repu_match_valide
False    1074482
True       13623
Name: count, dtype: int64
-----------------
df_ND1516_with_speaker repu_match_valide_net counts :
 repu_match_valide_net
False    1074482
True       13623
Name: count, dtype: int64


# Confrontation écart des cas vs ND — version factorisée

NOTE : reprend les fonctions `nettoyer_texte` et `count_lexical_outside_excl`
définies plus haut dans le notebook (section "Nettoyage du texte" / "Regex").
Ne pas exécuter cette section seule sans avoir exécuté ces cellules avant.

Objectif de la factorisation :
- éviter la duplication (pnum/id_syceron + snippets, chacun x2 pour
  tous/avec-speaker) qui existait déjà
- permettre d'ajouter facilement la version "texte brut" (non nettoyé) en
  plus de la version "texte nettoyé", sans dupliquer le code une 3e/4e fois
- centraliser les exports (nommage cohérent par configuration)

In [28]:
import pandas as pd
import re

P_NUMBER_RE = re.compile(r"#P?(\d+)", re.I)


def extract_pnum(url):
    if not url or pd.isna(url):
        return None
    url = str(url).strip()
    m = P_NUMBER_RE.search(url)
    return m.group(1) if m else None


# --- Chargement ---
df_extract_comp = pd.read_csv(
    "../data/interim/1_2_extract_15_16_concat.csv", low_memory=False
)
df_nd_comp = pd.read_csv(
    "../data/raw/test_regards_citoyens/ND15+16_interventions_hemicycle_rich.tsv",
    sep="\t",
    engine="python",
    on_bad_lines="warn",
)


In [29]:
# ==============================================================================
# Préparation commune : texte nettoyé, comptages brut/net, clés, masques speaker
# ==============================================================================


def preparer_corpus(df, col_texte):
    """
    Ajoute à df, à partir de `col_texte` :
    - `{col_texte}_net`                       : version nettoyée (nettoyer_texte)
    - `nb_repu_texte_brut` / `match_texte_brut` : comptage + match sur le texte non nettoyé
    - `nb_repu_texte_net`  / `match_texte_net`  : comptage + match sur le texte nettoyé

    Le nom de la colonne nettoyée est dérivé de `col_texte` (ex: "texte" ->
    "texte_net", "intervention" -> "intervention_net") plutôt que fixé en
    dur, pour éviter qu'une colonne issue de "intervention" ne s'appelle
    "texte_net" une fois nettoyée (confusion entre les deux corpus).

    Permet ensuite de choisir, au moment de la confrontation, quelle variante
    (texte_brut ou texte_net) utiliser, sans recalculer.
    """
    df = df.copy()
    col_texte_net = f"{col_texte}_net"
    df[col_texte_net] = df[col_texte].fillna("").apply(nettoyer_texte)

    df["nb_repu_texte_brut"] = df[col_texte].apply(count_lexical_outside_excl)
    df["match_texte_brut"] = df["nb_repu_texte_brut"] > 0

    df["nb_repu_texte_net"] = df[col_texte_net].apply(count_lexical_outside_excl)
    df["match_texte_net"] = df["nb_repu_texte_net"] > 0

    return df


df_extract_comp = preparer_corpus(df_extract_comp, "texte")
df_nd_comp = preparer_corpus(df_nd_comp, "intervention")
# -> colonnes nettoyées obtenues : df_extract_comp["texte_net"], df_nd_comp["intervention_net"]

# --- Clé pnum côté ND / harmonisation des types ---
df_nd_comp["pnum"] = pd.to_numeric(
    df_nd_comp["source"].apply(extract_pnum), errors="raise"
).astype("Int64")
df_extract_comp["id_syceron"] = df_extract_comp["id_syceron"].astype("Int64")

print("Doublons id_syceron :", df_extract_comp["id_syceron"].duplicated().sum())
print("Doublons pnum :", df_nd_comp["pnum"].duplicated().sum())


def masque_speaker_extract(df):
    return (
        df["id_acteur"].notna()
        | df["id_orateur"].notna()
        | df["nom_orateur"].fillna("").astype(str).str.strip().ne("")
    )


def masque_speaker_nd(df):
    return df["parlementaire"].fillna("").astype(str).str.strip().ne("") | df[
        "personnalite"
    ].fillna("").astype(str).str.strip().ne("")


df_extract_comp["a_un_speaker"] = masque_speaker_extract(df_extract_comp)
df_nd_comp["a_un_speaker"] = masque_speaker_nd(df_nd_comp)


Doublons id_syceron : 367
Doublons pnum : 281705


In [30]:
# ==============================================================================
# Fonctions génériques de confrontation
# ==============================================================================


def confrontation_pnum(df_extract, df_nd, match_col, label):
    """
    Compare les clés pnum (ND) / id_syceron (extract) : identifie les lignes
    ND absentes de l'extraction, restreint à celles avec mention valide, exporte.
    """
    cles_extract = set(df_extract["id_syceron"])
    df_nd_only = df_nd[~df_nd["pnum"].isin(cles_extract)]
    df_nd_only_match = df_nd_only[df_nd_only[match_col]]

    print(f"{'-' * 10} pnum vs id_syceron [{label}] {'-' * 10}")
    print(
        "Lignes ND :",
        df_nd.shape[0],
        "| absentes de l'extraction :",
        df_nd_only.shape[0],
    )
    print("Dont mention valide de République :", df_nd_only_match.shape[0])

    df_nd_only_match.to_csv(f"nd_pnum_absentes_de_extract_{label}.csv", index=False)
    return df_nd_only_match


def search_snippets_fast(source_df, source_text_col, big_target, snippet_len=150):
    df_out = source_df.copy()
    df_out["snippet"] = df_out[source_text_col].fillna("").str[:snippet_len]
    df_out["found"] = df_out["snippet"].apply(lambda s: bool(s) and (s in big_target))
    return df_out


PATTERNS_CHECK = {
    "parentheses": r"\([^)]*\)",
    "points_suspension": r"\.\.\.|…",  # "..." (trois points) ou le caractère unicode … (U+2026)
    # ajouter ici d'autres patterns si besoin (ex: "guillemets": r"[«»]") :
    "caractere_œ": r"œ",
    "mots_oe": (
        r"\b(?:"
        r"coeur|oeuvr\w*|voeux|voeu|Woerth|soeur|soeurs|"
        r"oeillères|manoeuvre\w*|oecuménisme|oeil"
        r")\b"
    ),  # explicite pour éviter tous mots contenants oe genre Éric Woerth
    "tiret_avant_ponctuation": r"– [,.]",
}


def ajouter_checks_patterns(df, col_texte_original, patterns=PATTERNS_CHECK):
    df = df.copy()
    cols_check = []
    for nom, pattern in patterns.items():
        col = f"check_{nom}"
        df[col] = df[col_texte_original].str.contains(pattern, regex=True, na=False)
        cols_check.append(col)
    # union logique (pas une somme) : une ligne qui a parenthèses ET points
    # de suspension ne compte qu'une fois ici
    df["check_au_moins_un"] = df[cols_check].any(axis=1)
    return df


def print_checks_patterns(df, patterns=PATTERNS_CHECK):
    print("  dont patterns (dans le texte original) :")

    for nom in patterns:
        print(f"    dont avec {nom} :", df[f"check_{nom}"].sum())
    print("  dont avec au moins un pattern (croisé) :", df["check_au_moins_un"].sum())


def confrontation_snippets(
    df_extract,
    df_nd,
    text_col_extract,
    text_col_nd,
    match_col_extract,
    match_col_nd,
    label,
    col_texte_original_extract="texte",
    col_texte_original_nd="intervention",
    snippet_len=150,
):
    """
    Compare par snippets de texte (dans les deux sens), restreint aux lignes
    avec mention valide de République, exporte les deux sens + ajoute le
    check parenthèses.

    IMPORTANT : le filtrage est cumulatif et entièrement local à cet appel.
    `df_extract` / `df_nd` reçus ici sont déjà le sous-ensemble "speaker"
    choisi par l'appelant (boucle de configuration) ; on y ajoute ici le
    filtre "match repu valide". Le "big" de comparaison (big_extract_snippet
    / big_nd_snippet) est reconstruit à chaque appel à partir de ce double
    filtre : aucune variable partagée entre itérations de la boucle, donc
    pas de risque de mélanger les configs (ex: le big de "texte_net_speaker"
    ne peut pas contenir de texte de la variante "texte_brut" ou d'un run
    "sans filtre speaker").
    """
    df_nd_snippet = df_nd[df_nd[match_col_nd]].copy()
    df_extract_snippet = df_extract[df_extract[match_col_extract]].copy()

    print(f"{'-' * 10} snippets [{label}] {'-' * 10}")
    print("Lignes ND avec mention repu valide       :", df_nd_snippet.shape[0])
    print("Lignes extract avec mention repu valide  :", df_extract_snippet.shape[0])

    # Reconstruit à chaque appel (pas de cache/variable globale) : reflète
    # exactement le double filtre speaker + match ci-dessus pour CETTE config.
    big_extract_snippet = " ".join(df_extract_snippet[text_col_extract].tolist())
    big_nd_snippet = " ".join(df_nd_snippet[text_col_nd].tolist())

    # nd abs dans extract
    res_nd_in_extract = search_snippets_fast(
        df_nd_snippet, text_col_nd, big_extract_snippet, snippet_len
    )
    nd_absent = res_nd_in_extract[~res_nd_in_extract["found"]]
    nd_absent = ajouter_checks_patterns(nd_absent, col_texte_original_nd)
    print(
        "ND (repu) introuvable dans extract :",
        nd_absent.shape[0],
        "/",
        len(res_nd_in_extract),
    )
    print_checks_patterns(nd_absent)
    nd_absent.to_csv(f"nd_snippets_absent_de_extract_{label}.csv", index=False)

    # extract abs dans nd
    res_extract_in_nd = search_snippets_fast(
        df_extract_snippet, text_col_extract, big_nd_snippet, snippet_len
    )
    extract_absent = res_extract_in_nd[~res_extract_in_nd["found"]]
    extract_absent = ajouter_checks_patterns(extract_absent, col_texte_original_extract)
    print(
        "Extract (repu) introuvable dans ND :",
        extract_absent.shape[0],
        "/",
        len(res_extract_in_nd),
    )
    print_checks_patterns(extract_absent)
    extract_absent.to_csv(f"extract_snippets_absent_de_nd_{label}.csv", index=False)

    return nd_absent, extract_absent

In [31]:
# ==============================================================================
# Boucle sur les configurations (texte brut/net x tous/avec-orateur)
# ==============================================================================

configs_texte = {
    "texte_brut": dict(
        col_extract="texte", col_nd="intervention", match="match_texte_brut"
    ),
    "texte_net": dict(
        col_extract="texte_net", col_nd="intervention_net", match="match_texte_net"
    ),
}

resultats = {}

for variante, cfg in configs_texte.items():
    for filtre_speaker in (False, True):
        label = variante + ("_speaker" if filtre_speaker else "")

        df_extract_sub = (
            df_extract_comp[df_extract_comp["a_un_speaker"]]
            if filtre_speaker
            else df_extract_comp
        )
        df_nd_sub = (
            df_nd_comp[df_nd_comp["a_un_speaker"]] if filtre_speaker else df_nd_comp
        )

        print("=" * 60)
        print(f"CONFIGURATION : {label}")
        print("=" * 60)

        nd_only_match = confrontation_pnum(
            df_extract_sub, df_nd_sub, match_col=cfg["match"], label=label
        )

        # La comparaison par snippets repose sur une égalité de sous-chaîne
        # EXACTE entre deux corpus qui n'ont pas la même structure brute
        # (balises HTML, apostrophes, espaces...). Sans nettoyer_texte()
        # appliqué aux deux côtés, elle produit un taux d'"introuvable"
        # artificiellement proche de 100% qui ne reflète aucun vrai écart de
        # contenu. On ne la lance donc que sur la variante "texte_net" (seule
        # variante où les deux textes sont normalisés de façon comparable).
        if variante == "texte_net":
            nd_absent, extract_absent = confrontation_snippets(
                df_extract_sub,
                df_nd_sub,
                text_col_extract=cfg["col_extract"],
                text_col_nd=cfg["col_nd"],
                match_col_extract=cfg["match"],
                match_col_nd=cfg["match"],
                label=label,
            )
        else:
            nd_absent, extract_absent = None, None

        resultats[label] = dict(
            pnum_absent=nd_only_match,
            nd_snippets_absent=nd_absent,
            extract_snippets_absent=extract_absent,
        )


CONFIGURATION : texte_brut
---------- pnum vs id_syceron [texte_brut] ----------
Lignes ND : 1391207 | absentes de l'extraction : 45410
Dont mention valide de République : 130
CONFIGURATION : texte_brut_speaker
---------- pnum vs id_syceron [texte_brut_speaker] ----------
Lignes ND : 1088105 | absentes de l'extraction : 841
Dont mention valide de République : 81
CONFIGURATION : texte_net
---------- pnum vs id_syceron [texte_net] ----------
Lignes ND : 1391207 | absentes de l'extraction : 45410
Dont mention valide de République : 130
---------- snippets [texte_net] ----------
Lignes ND avec mention repu valide       : 13675
Lignes extract avec mention repu valide  : 13095
ND (repu) introuvable dans extract : 1229 / 13675
  dont patterns (dans le texte original) :
    dont avec parentheses : 32
    dont avec points_suspension : 617
    dont avec caractere_œ : 33
    dont avec mots_oe : 234
    dont avec tiret_avant_ponctuation : 145
  dont avec au moins un pattern (croisé) : 851
Extract 

In [32]:
# POUR LA SCIENCE : SI VEUT FAIRE TOURNER SNIPPET SUR BRUT POUR OBSERVER LES DIFF
# (même si forcément résultat artificiellement mauvais, car txt pas normalisé)


# # ==============================================================================
# # Boucle sur les configurations (texte brut/net x tous/avec-orateur)
# # ==============================================================================

# configs_texte = {
#     "texte_brut": dict(
#         col_extract="texte", col_nd="intervention", match="match_texte_brut"
#     ),
#     "texte_net": dict(
#         col_extract="texte_net", col_nd="intervention_net", match="match_texte_net"
#     ),
# }

# resultats = {}

# for variante, cfg in configs_texte.items():
#     for filtre_speaker in (False, True):
#         label = variante + ("_speaker" if filtre_speaker else "")

#         df_extract_sub = (
#             df_extract_comp[df_extract_comp["a_un_speaker"]]
#             if filtre_speaker
#             else df_extract_comp
#         )
#         df_nd_sub = (
#             df_nd_comp[df_nd_comp["a_un_speaker"]] if filtre_speaker else df_nd_comp
#         )

#         print("=" * 60)
#         print(f"CONFIGURATION : {label}")
#         print("=" * 60)

#         nd_only_match = confrontation_pnum(
#             df_extract_sub, df_nd_sub, match_col=cfg["match"], label=label
#         )

#         # La comparaison par snippets repose sur une égalité de sous-chaîne
#         # EXACTE entre deux corpus qui n'ont pas la même structure brute
#         # (balises HTML, apostrophes, espaces...). Sans nettoyer_texte()
#         # appliqué aux deux côtés, elle produit un taux d'"introuvable"
#         # artificiellement proche de 100% qui ne reflète aucun vrai écart de
#         # contenu. On ne la lance donc que sur la variante "texte_net" (seule
#         # variante où les deux textes sont normalisés de façon comparable).
#         if variante == "texte_net" or "texte_brut":
#             nd_absent, extract_absent = confrontation_snippets(
#                 df_extract_sub,
#                 df_nd_sub,
#                 text_col_extract=cfg["col_extract"],
#                 text_col_nd=cfg["col_nd"],
#                 match_col_extract=cfg["match"],
#                 match_col_nd=cfg["match"],
#                 label=label,
#             )
#         else:
#             nd_absent, extract_absent = None, None

#         resultats[label] = dict(
#             pnum_absent=nd_only_match,
#             nd_snippets_absent=nd_absent,
#             extract_snippets_absent=extract_absent,
#         )


## Tests et fuzzy

In [50]:
# ==============================================================================
# Relance ciblée d'une configuration pour fuzzy
# ==============================================================================

variante = "texte_net"
filtre_speaker = True
label_fuzzy = variante + ("_speaker" if filtre_speaker else "")

cfg = configs_texte[variante]

df_extract_sub = (
    df_extract_comp[df_extract_comp["a_un_speaker"]]
    if filtre_speaker
    else df_extract_comp
)

df_nd_sub = (
    df_nd_comp[df_nd_comp["a_un_speaker"]]
    if filtre_speaker
    else df_nd_comp
)

# Rejoue exactement la confrontation snippets
nd_absent, extract_absent = confrontation_snippets(
    df_extract_sub,
    df_nd_sub,
    text_col_extract=cfg["col_extract"],
    text_col_nd=cfg["col_nd"],
    match_col_extract=cfg["match"],
    match_col_nd=cfg["match"],
    label=label_fuzzy,
)

---------- snippets [texte_net_speaker] ----------
Lignes ND avec mention repu valide       : 13623
Lignes extract avec mention repu valide  : 13095
ND (repu) introuvable dans extract : 1209 / 13623
  dont patterns (dans le texte original) :
    dont avec parentheses : 32
    dont avec points_suspension : 617
    dont avec caractere_œ : 33
    dont avec mots_oe : 234
    dont avec tiret_avant_ponctuation : 145
  dont avec au moins un pattern (croisé) : 851
Extract (repu) introuvable dans ND : 2579 / 13095
  dont patterns (dans le texte original) :
    dont avec parentheses : 2033
    dont avec points_suspension : 1050
    dont avec caractere_œ : 492
    dont avec mots_oe : 7
    dont avec tiret_avant_ponctuation : 0
  dont avec au moins un pattern (croisé) : 2415


In [51]:
df_extract_ref = df_extract_sub[df_extract_sub[cfg["match"]]]
df_nd_ref = df_nd_sub[df_nd_sub[cfg["match"]]]

In [53]:
# ==============================================================================
# Diagnostic fuzzy post-confrontation des snippets absents
# ==============================================================================

from rapidfuzz import process, fuzz


def fuzzy_snippet_diagnostic(
    df_absent,
    source_text_col,
    target_df,
    target_text_col,
    snippet_len=150,
    score_min=95,
):
    """
    Cherche un équivalent fuzzy pour des snippets déclarés absents
    après recherche exacte.

    Usage :
    - diagnostic uniquement
    - ne remplace pas la confrontation exacte

    Retourne le dataframe enrichi avec :
    - snippet_fuzzy
    - fuzzy_match
    - fuzzy_score
    - fuzzy_probable
    """

    if df_absent is None or df_absent.empty:
        return df_absent

    out = df_absent.copy()

    # ------------------------------------------------------------------
    # Corpus cible réduit aux snippets
    # ------------------------------------------------------------------

    target_snippets = (
        target_df[target_text_col]
        .fillna("")
        .astype(str)
        .str[:snippet_len]
        .drop_duplicates()
        .tolist()
    )

    # ------------------------------------------------------------------
    # Snippets à tester
    # ------------------------------------------------------------------

    out["snippet_fuzzy"] = (
        out[source_text_col]
        .fillna("")
        .astype(str)
        .str[:snippet_len]
    )

    fuzzy_matches = []

    for snippet in out["snippet_fuzzy"]:

        if not snippet:
            fuzzy_matches.append(("", 0))
            continue

        best = process.extractOne(
            snippet,
            target_snippets,
            scorer=fuzz.ratio,
        )

        if best is None:
            fuzzy_matches.append(("", 0))
        else:
            fuzzy_matches.append(
                (
                    best[0],  # texte cible
                    best[1],  # score
                )
            )

    out["fuzzy_match"] = [x[0] for x in fuzzy_matches]
    out["fuzzy_score"] = [x[1] for x in fuzzy_matches]

    out["fuzzy_probable"] = (
        out["fuzzy_score"] >= score_min
    )

    return out.sort_values(
        "fuzzy_score",
        ascending=False,
    )


# ==============================================================================
# Paramètres du diagnostic
# ==============================================================================

label_fuzzy = "texte_net_speaker"

cfg_fuzzy = configs_texte["texte_net"]

# récupérer les faux négatifs exacts
df_extract_ref = df_extract_sub[df_extract_sub[cfg["match"]]]
df_nd_ref = df_nd_sub[df_nd_sub[cfg["match"]]]


# ==============================================================================
# Corpus de référence
# ==============================================================================

df_extract_ref = df_extract_comp[
    df_extract_comp[cfg_fuzzy["match"]]
]

df_nd_ref = df_nd_comp[
    df_nd_comp[cfg_fuzzy["match"]]
]


# ==============================================================================
# ND absent dans extract -> recherche fuzzy côté extract
# ==============================================================================

diag_nd = fuzzy_snippet_diagnostic(
    df_absent=nd_absent,
    source_text_col="intervention_net",
    target_df=df_extract_ref,
    target_text_col="texte_net",
    snippet_len=150,
    score_min=95,
)

print(
    "ND absents analysés fuzzy :",
    len(diag_nd),
)

if diag_nd is not None:
    print(
        "Dont fuzzy probable (>95) :",
        diag_nd["fuzzy_probable"].sum(),
    )

    diag_nd.to_csv(
        f"diagnostic_fuzzy_nd_vs_extract_{label_fuzzy}.csv",
        index=False,
    )


# ==============================================================================
# Extract absent dans ND -> recherche fuzzy côté ND
# ==============================================================================

diag_extract = fuzzy_snippet_diagnostic(
    df_absent=extract_absent,
    source_text_col="texte_net",
    target_df=df_nd_ref,
    target_text_col="intervention_net",
    snippet_len=150,
    score_min=95,
)

print(
    "Extract absents analysés fuzzy :",
    len(diag_extract),
)

if diag_extract is not None:
    print(
        "Dont fuzzy probable (>95) :",
        diag_extract["fuzzy_probable"].sum(),
    )

    diag_extract.to_csv(
        f"diagnostic_fuzzy_extract_vs_nd_{label_fuzzy}.csv",
        index=False,
    )


# ==============================================================================
# Aperçu des cas les plus proches
# ==============================================================================

display(
    diag_nd[
        [
            "intervention_net",
            "fuzzy_match",
            "fuzzy_score",
        ]
    ].head(30)
)

display(
    diag_extract[
        [
            "texte_net",
            "fuzzy_match",
            "fuzzy_score",
        ]
    ].head(30)
)

ND absents analysés fuzzy : 1209
Dont fuzzy probable (>95) : 966
Extract absents analysés fuzzy : 2579
Dont fuzzy probable (>95) : 980


,intervention_net,fuzzy_match,fuzzy_score
225930,… et qu'aucune règle de notre régime républica...,…et qu'aucune règle de notre régime républicai...,99.665552
1205885,Vous ânonnez des solutions qui n'existent que ...,Vous ânonnez des solutions qui n'existent que ...,99.665552
1252195,"…nous souhaitons, de nouveau, vous rappeler ce...","…nous souhaitons, de nouveau, vous rappeler ce...",99.663300
670042,… aux ennemis de la patrie républicaine : nous...,…aux ennemis de la patrie républicaine : nous ...,99.661017
545692,Je ne suis pas opposé à la police – nous avons...,Je ne suis pas opposé à la police – nous avons...,99.649123
31143,… car il ne saurait être question qu'une affai...,…car il ne saurait être question qu'une affair...,99.646643
729605,"… qui sera retiré au club, signifiant pour lui...","…qui sera retiré au club, signifiant pour lui ...",99.644128
75273,"… déjà profondes, qui minent durement notre pa...","…déjà profondes, qui minent durement notre pac...",99.644128
1203859,La Déclaration des droits de l'homme et du cit...,La Déclaration des droits de l'homme et du cit...,99.644128
719018,… consubstantielle à l'idée de République fran...,…consubstantielle à l'idée de République franç...,99.638989


,texte_net,fuzzy_match,fuzzy_score
694018,Pro patria vigilant – « Ils veillent pour la p...,Pro patria vigilant – « Ils veillent pour la p...,100.000000
709598,Comment on a laissé l'islamisme pénétrer l'éco...,Comment on a laissé l'islamisme pénétrer l'éco...,100.000000
818537,Vous ânonnez des solutions qui n'existent que ...,Vous ânonnez des solutions qui n'existent que ...,99.665552
32946,…et qu'aucune règle de notre régime républicai...,… et qu'aucune règle de notre régime républica...,99.665552
1019249,"…nous souhaitons, de nouveau, vous rappeler ce...","…nous souhaitons, de nouveau, vous rappeler ce...",99.663300
571548,…aux ennemis de la patrie républicaine : nous ...,… aux ennemis de la patrie républicaine : nous...,99.661017
480592,Je ne suis pas opposé à la police – nous avons...,Je ne suis pas opposé à la police – nous avons...,99.649123
61497,…car il ne saurait être question qu'une affair...,… car il ne saurait être question qu'une affai...,99.646643
816940,La Déclaration des droits de l'homme et du cit...,La Déclaration des droits de l'homme et du cit...,99.644128
622882,"…qui sera retiré au club, signifiant pour lui ...","… qui sera retiré au club, signifiant pour lui...",99.644128
